# Player Feedback Radar — Yorum Etiketleme (Colab)

Açık kaynak bir dil modeliyle App Store yorumlarını `taxonomy.md` kategorilerine göre etiketler.

**Başlamadan önce**
1. Üst menü → **Runtime → Change runtime type → T4 GPU** seç.
2. Google Drive'ında `player-feedback-radar` adında bir klasör aç ve içine `reviews.csv` dosyasını yükle.
3. Hücreleri sırayla çalıştır (veya **Runtime → Run all**).

Oturum kapanırsa sorun değil: notebook'u tekrar çalıştırdığında, `labels_<sürüm>.csv` içinde zaten etiketlenmiş yorumları atlar ve kaldığı yerden devam eder.

## 1. GPU ve Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/player-feedback-radar'
REVIEWS_PATH = f'{BASE}/reviews.csv'
# LABELS_PATH ayarlar hücresinde PROMPT_VERSION'a göre belirlenir
assert os.path.exists(REVIEWS_PATH), f'{REVIEWS_PATH} bulunamadı — reviews.csv dosyasını Drive klasörüne yükle.'
print('Hazır:', REVIEWS_PATH)

## 2. Kurulum

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

## 3. Ayarlar

- `MODEL_ID`: Varsayılan **Qwen2.5-7B-Instruct** (4-bit) — 3B modele göre belirgin şekilde daha isabetli, T4'te 5.400 yorum ≈ 2–3 saat.
  Sadece hızlı deneme için `Qwen/Qwen2.5-3B-Instruct` ve `LOAD_4BIT = False` kullanılabilir.
- `LIMIT`: Deneme için bir sayı ver (her oyundan eşit pay alınır); her şey yolundaysa `None` yapıp tamamını çalıştır.

In [ ]:
MODEL_ID   = 'Qwen/Qwen2.5-7B-Instruct'
LOAD_4BIT  = True       # 3B model için False
BATCH_SIZE = 4          # bellek yetmezse otomatik olarak yarıya iner
MAX_CHARS  = 800        # çok uzun yorumları kırp
LIMIT      = 80         # deneme: her oyundan 20 yorum; tamamı için None
PROMPT_VERSION = 'v0.4' # prompt değişince artır -> sonuçlar ayrı dosyaya yazılır
LABELS_PATH = f'{BASE}/labels_{PROMPT_VERSION}.csv'
print('Çıktı dosyası:', LABELS_PATH)
SAVE_EVERY = 5          # kaç batch'te bir Drive'a kaydetsin

## 4. Etiket şeması ve prompt

In [ ]:
CATEGORIES = {
    'difficulty':      'levels too hard/easy, stuck on a level, feels pay-to-win, unfair difficulty',
    'ads':             'in-game ads: too frequent, too long, forced/unskippable, ads after paying',
    'rewarded_ads':    'opt-in reward videos: wants more of them, daily cap/limit, rewards not given',
    'misleading_ads':  'the game is different from its advertisements/commercials ("nothing like the ads")',
    'monetization':    'prices, bundles, passes, spending pressure, "money grab", purchase problems',
    'lives_energy':    'running out of lives/energy, waiting timers',
    'bug_performance': 'crashes, freezes, lag, lost progress, battery, loading issues',
    'content':         'wants new levels, ran out of content, repetitive levels; for word games: missing or wrong words in the dictionary',
    'events':          'time-limited events, mini-games, side modes, passes: too many, too few, unfair, confusing',
    'social_pvp':      'teams, opponents, matchmaking, bots, cheating, leaderboards',
    'ux_ui':           'interface, controls, readability, tutorial, confusing design',
    'praise':          'general praise, fun, addictive in a positive way, relaxing',
    'feature_request': 'a concrete suggestion for a new feature or change',
}

cat_lines = '\n'.join(f'- {k}: {v}' for k, v in CATEGORIES.items())

# Talimatlar ve kategori listesi sistem mesajında BİR KEZ yer alır; her yorum mesajı kısa kalır (GPU belleği için önemli)
SYSTEM_PROMPT = (
    'You label mobile game App Store reviews for product analysis. '
    'Reviews may be in English, Turkish, German or other languages.\n\n'
    'Categories (choose 1-3 that clearly apply; use [] only if none fit):\n'
    f'{cat_lines}\n\n'
    'Rules: be strict, only choose a category the review explicitly talks about. '
    'Players often write "adds" when they mean "ads". '
    'Any comment about fun or enjoyment is praise (ux_ui is ONLY for interface, controls, tutorial, '
    'pop-ups or rating prompts). "Rigged", "fixed", "impossible without buying" means difficulty. '
    'Repeated or recycled levels mean content. Use bug_performance only for technical faults.\n\n'
    'Also give: s = sentiment ("pos", "neu" or "neg"); '
    'v = severity for negative reviews (1 minor annoyance, 2 hurts the experience, 3 quit or about to quit; 0 if not negative); '
    'ch = 1 only if the reviewer explicitly says they deleted, uninstalled or quit, else 0.\n\n'
    'Return ONLY one line of compact JSON, exactly like: {"c":["category"],"s":"neg","v":2,"ch":0}'
)

REVIEW_TEMPLATE = 'Game: {game}\nStars: {rating}/5\nTitle: {title}\nReview: {text}'
# Çözülmüş örnekler (few-shot): küçük modellerin kategori isabetini belirgin artırır
FEW_SHOT = [
    (dict(game='Puzzle Game', rating=5, title='Fun', text='Entretenido'),
     '{"c":["praise"],"s":"pos","v":0,"ch":0}'),
    (dict(game='Word Game', rating=5, title='Love it', text='Making long powerful words is so satisfying!'),
     '{"c":["praise"],"s":"pos","v":0,"ch":0}'),
    (dict(game='Match Game', rating=1, title='Rigged', text='Stop asking me to rate the game. Levels are rigged so you have to buy boosters.'),
     '{"c":["ux_ui","difficulty","monetization"],"s":"neg","v":2,"ch":0}'),
    (dict(game='Word Game', rating=2, title='Dictionary', text="It doesn't accept real words like 'ain't' and 'tsar'. So annoying."),
     '{"c":["content"],"s":"neg","v":2,"ch":0}'),
    (dict(game='Puzzle Game', rating=1, title='Sıkıcı', text='Aynı bölümler tekrar tekrar geliyor, süre de kısaldı, sıkıldım.'),
     '{"c":["content","difficulty"],"s":"neg","v":2,"ch":0}'),
    (dict(game='Match Game', rating=1, title='Fake', text='Nothing like the adds on Instagram. Deleted.'),
     '{"c":["misleading_ads"],"s":"neg","v":3,"ch":1}'),
    (dict(game='Puzzle Game', rating=4, title='More videos', text='Great game, I just wish I could watch more videos for extra moves.'),
     '{"c":["rewarded_ads","feature_request","praise"],"s":"pos","v":0,"ch":0}'),
    (dict(game='Match Game', rating=3, title='Donuyor', text='Oyunu seviyorum ama son güncellemeden beri sürekli donuyor ve ilerlemem silindi.'),
     '{"c":["bug_performance"],"s":"neg","v":2,"ch":0}'),
]

def build_messages(row):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for ex, answer in FEW_SHOT:
        msgs.append({'role': 'user', 'content': REVIEW_TEMPLATE.format(**ex)})
        msgs.append({'role': 'assistant', 'content': answer})
    msgs.append({'role': 'user', 'content': build_prompt(row)})
    return msgs

print(SYSTEM_PROMPT)
print('Few-shot örnek sayısı:', len(FEW_SHOT))

## 5. Yardımcı fonksiyonlar (ayrıştırma ve kaldığı yerden devam)

In [ ]:
import json, re
import pandas as pd

LABEL_COLS = ['review_hash', 'categories', 'sentiment', 'severity', 'churn_signal', 'parse_ok', 'model']
VALID_S = {'pos', 'neu', 'neg'}

def parse_label(raw):
    # Model çıktısından JSON'u ayıklar ve doğrular; bozuksa parse_ok=False döner.
    m = re.search(r'\{.*\}', raw, re.S)
    try:
        d = json.loads(m.group(0)) if m else None
    except Exception:
        d = None
    if not isinstance(d, dict):
        return {'categories': '', 'sentiment': '', 'severity': '', 'churn_signal': '', 'parse_ok': False}
    cats = d.get('c', [])
    if isinstance(cats, str):
        cats = [cats]
    cats = [c for c in cats if c in CATEGORIES][:3]
    s = d.get('s') if d.get('s') in VALID_S else ''
    try:
        v = int(d.get('v', 0)); v = v if v in (0, 1, 2, 3) else 0
    except Exception:
        v = 0
    try:
        ch = 1 if int(d.get('ch', 0)) == 1 else 0
    except Exception:
        ch = 0
    if s == 'neg' and v == 0:
        v = 1              # olumsuz yorumun şiddeti en az 1
    if s != 'neg':
        v = 0
    return {'categories': '|'.join(cats), 'sentiment': s, 'severity': v,
            'churn_signal': ch, 'parse_ok': bool(s)}

def load_done():
    if os.path.exists(LABELS_PATH):
        done = pd.read_csv(LABELS_PATH)
        return done, set(done['review_hash'])
    return pd.DataFrame(columns=LABEL_COLS), set()

def build_prompt(row):
    text = str(row['text'])[:MAX_CHARS]
    return REVIEW_TEMPLATE.format(game=row['app_name'], rating=row['rating'],
                                title=str(row['title'])[:150], text=text)

# Hızlı test
print(parse_label('{"c":["ads","monetization"],"s":"neg","v":3,"ch":1}'))
print(parse_label('Sure! {"c":["unknown","praise"],"s":"pos","v":0,"ch":0}'))
print(parse_label('bozuk çıktı'))

## 6. Modeli yükle

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

kwargs = {'device_map': 'auto'}
if LOAD_4BIT:
    kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
else:
    kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()
print('Model yüklendi:', MODEL_ID)

## 7. Etiketle

In [ ]:
from tqdm.auto import tqdm

reviews = pd.read_csv(REVIEWS_PATH)
reviews['date'] = pd.to_datetime(reviews['date'], utc=True, errors='coerce')
reviews = reviews.sort_values('date', ascending=False)   # en yeniler önce: oturum kesilirse güncel veri hazır olur

done_df, done = load_done()
todo = reviews[~reviews['review_hash'].isin(done)]
if LIMIT:
    per_app = max(1, LIMIT // reviews['app_key'].nunique())
    todo = todo.groupby('app_key', group_keys=False).head(per_app)   # deneme: her oyundan eşit pay
print(f'Zaten etiketli: {len(done)} | Bu çalıştırmada: {len(todo)}')

def _generate_once(batch_rows):
    prompts = [tok.apply_chat_template(build_messages(r), tokenize=False, add_generation_prompt=True)
               for r in batch_rows]
    enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=60, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)

def generate(batch_rows):
    # Bellek yetmezse grubu ikiye bölüp tekrar dener (tek yoruma kadar)
    try:
        return _generate_once(batch_rows)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch_rows) == 1:
            print('  ! Tek yorum bile sığmadı, atlanıyor')
            return ['']
        mid = len(batch_rows) // 2
        return generate(batch_rows[:mid]) + generate(batch_rows[mid:])

buffer = []
rows = todo.to_dict('records')
batches = [rows[i:i + BATCH_SIZE] for i in range(0, len(rows), BATCH_SIZE)]

def flush():
    global done_df, buffer
    if buffer:
        done_df = pd.concat([done_df, pd.DataFrame(buffer)], ignore_index=True)
        done_df.to_csv(LABELS_PATH, index=False)
        buffer = []

for i, batch in enumerate(tqdm(batches)):
    for r, raw in zip(batch, generate(batch)):
        buffer.append({'review_hash': r['review_hash'], **parse_label(raw), 'model': f'{MODEL_ID}|{PROMPT_VERSION}'})
    if (i + 1) % SAVE_EVERY == 0:
        flush()
flush()

print('Toplam etiketli:', len(done_df), '| Ayrıştırma başarısı:', round(done_df['parse_ok'].mean(), 3))

## 8. Hızlı kontrol

In [ ]:
labels = pd.read_csv(LABELS_PATH)
df = reviews.merge(labels, on='review_hash')

print('Duygu dağılımı:\n', df['sentiment'].value_counts(normalize=True).round(2), '\n')
exploded = df.assign(cat=df['categories'].fillna('').str.split('|')).explode('cat')
exploded = exploded[exploded['cat'] != ''].reset_index(drop=True)
print('Oyun x kategori (yorumların yüzdesi):')
print((pd.crosstab(exploded['app_name'], exploded['cat'])
       .div(df.groupby('app_name').size(), axis=0) * 100).round(1).to_string())

# Örnek çıktılar — gözle kontrol et
pd.set_option('display.max_colwidth', 120)
df[['app_name', 'rating', 'text', 'categories', 'sentiment', 'severity', 'churn_signal']].sample(10, random_state=1)

## 9. Doğrulama örneklemi

100 yorumluk dengeli bir örneklem çıkarır. Bu dosyayı indirip `human_categories` ve `human_sentiment`
sütunlarını elle doldurursak modelin doğruluk oranını hesaplayıp README'ye yazarız.

In [ ]:
picked = (df.groupby('app_name')['review_hash']
            .apply(lambda s: s.sample(min(len(s), 25), random_state=42)))
sample = df[df['review_hash'].isin(picked.values)]
sample = sample[['review_hash', 'app_name', 'rating', 'title', 'text', 'categories', 'sentiment']]
sample['human_categories'] = ''
sample['human_sentiment'] = ''
sample.to_csv(f'{BASE}/validation_sample.csv', index=False)
print('Kaydedildi:', f'{BASE}/validation_sample.csv', '| satır:', len(sample))